## Baseline (no fine-tuning) — Task 2 (Structured Entity Extraction)

This notebook is an **exact replica** of [llm_fine_tuning_LORA_task2.ipynb](llm_fine_tuning_LORA_task2.ipynb) with **one difference: there is no training**. The base **`meta-llama/Meta-Llama-3.1-8B`** model is loaded and used *as-is* to make predictions on the **same validation set** the fine-tuned run is evaluated on, so the two sets of metrics are directly comparable. It mirrors the Task 1 baseline notebook, [llama_3.1_task_1_no_fine_tune.ipynb](llama_3.1_task_1_no_fine_tune.ipynb).

Task 2 = given a contract clause excerpt, extract one specific fact from it and return it as a **valid JSON object** with the category as its single key (a list of strings for multi-values, `null` when absent) across the 9 entity categories.

**Why this baseline matters:** the fine-tuned run measures two skills at once — extraction/normalization accuracy and strict-JSON format discipline. Running the **un-fine-tuned** base model through the *identical* evaluation (JSON validity → exact match → token-level F1, per category — see [TASK2_EVALUATION_METRICS.md](docs/task_2/TASK2_EVALUATION_METRICS.md)) tells us what the model can do out-of-the-box, which is the reference point fine-tuning is measured against. The expected headline delta is on **JSON validity**.

The data pipeline (load → normalize answers → build examples → contract-level 80/20 split → sanity checks → save JSONL) is byte-for-byte identical to the fine-tuning notebook (same `train_test_split(..., random_state=42)`), which guarantees the validation set here is exactly the one used there. All artifacts are written to a **separate `no_finetune_baseline_task2/` subdirectory** so they never overwrite the fine-tuned run's outputs (or the Task 1 baseline's `no_finetune_baseline/`).


In [ ]:
# --- Pin to a single GPU BEFORE torch is imported anywhere ---
# Kaggle "GPU T4 x2" exposes 2 GPUs. device_map="auto" then shards the model
# across cuda:0/cuda:1. At loss time TRL's _chunked_cross_entropy_loss builds the
# label mask on cuda:0 while the final hidden_states/lm_head live on cuda:1 ->
# "indices should be either on cpu or on the same device as the indexed tensor
# (cuda:1)". An 8B model in 4-bit (~5-6 GB) fits in ONE T4 (16 GB), so hide GPU 1.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Kaggle: install the library versions this notebook expects (no-op locally) ---
# The Kaggle base image ships older trl/peft; pin trl 1.x so SFTConfig,
# completion_only_loss and processing_class are available.
if os.path.exists("/kaggle"):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==4.55.4",   # MUST be <4.56: the 4.56 "core_model_loading"
                                              # threaded loader breaks bnb 4-bit -> full fp16 load -> OOM
                    "bitsandbytes==0.46.1",
                    "accelerate==1.7.0",
                    "peft==0.15.2",
                    "trl==0.20.0",
                    "datasets"], check=True)



In [ ]:
import sys; print("UTF-8 mode:", sys.flags.utf8_mode)

# Step 1 : Load data (cleaned master_clauses file from CUAD)

In [ ]:
import pandas as pd
import json
from pathlib import Path
import csv
import re
from sklearn.model_selection import train_test_split

In [ ]:
# --- Environment config: run unchanged locally OR on Kaggle ---
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()

if ON_KAGGLE:
    # Read-only mounted dataset (matches dataset-metadata.json id slug + folder)
    DATA_DIR = Path("/kaggle/input/cuad-master-clauses-cleaned")
    # Only this dir is writable AND persisted as kernel output:
    WORK_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path(os.getenv("DATA_DIR", "data")) / "CUAD_v1"
    WORK_DIR = Path(".")

print(f"ON_KAGGLE={ON_KAGGLE} | DATA_DIR={DATA_DIR} | WORK_DIR={WORK_DIR}")

# All artifacts from THIS (no-fine-tune) notebook go under a dedicated subdirectory
# so they never overwrite the fine-tuned Task 2 run's outputs (eval_metrics.json,
# cuad/, ...) nor the Task 1 baseline's no_finetune_baseline/.
OUTPUT_DIR = WORK_DIR / "no_finetune_baseline_task2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"OUTPUT_DIR={OUTPUT_DIR}")


In [ ]:
CUAD_PATH = DATA_DIR   # set by the environment-config cell
# Step 1: load the cleaned CSV.
#CSV_NAME = 'master_clauses_cleaned_sampled.csv'   ~100 records, quick iteration
CSV_NAME = 'master_clauses_cleaned.csv'        # full dataset

# Robustly locate the CSV. On Kaggle the dataset may mount under a different folder
# than expected (or not be attached at all) - search /kaggle/input as a fallback.
MASTER_CLAUSES_PATH = CUAD_PATH / CSV_NAME
if not MASTER_CLAUSES_PATH.exists():
    search_root = Path('/kaggle/input') if ON_KAGGLE else CUAD_PATH
    matches = list(search_root.rglob(CSV_NAME)) if search_root.exists() else []
    if matches:
        MASTER_CLAUSES_PATH = matches[0]
        print(f'Resolved CSV via fallback search: {MASTER_CLAUSES_PATH}')
    else:
        print(f'Could NOT find {CSV_NAME} under {search_root}. Actually mounted:')
        listing = sorted(search_root.rglob('*'))[:50] if search_root.exists() else []
        for _p in listing:
            print('   ', _p)
        if not listing:
            print('   (nothing — the dataset is not attached to this kernel)')
        raise FileNotFoundError(f'{CSV_NAME} not found under {search_root}')

# Read the file manually using the CSV module to handle inconsistencies
data = []
with open(MASTER_CLAUSES_PATH, 'r', encoding='utf-8', errors='replace') as f:
    reader = csv.DictReader(f)
    for row in reader:
        data.append(row)

df = pd.DataFrame(data)
print(f'Data Loaded Successfully from {MASTER_CLAUSES_PATH}')
print(f'Total Contracts: {len(df)}')
print(df.head(3))

In [ ]:
df.head(3)

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove special characters but keep spaces
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

# Clean column names
df.columns = [clean_text(col).strip() for col in df.columns]
for col in df.columns:
    print(col)

In [ ]:
new_columns = {}
for col in df.columns:
    print(f"Processing column: '{col}'")
    if "Answer" in col:
            # Remove "Answer" from the string and append "_Answer" at the end
            new_columns[col] = f"{col.replace('Answer', '').strip()}_Answer"

df = df.rename(columns=new_columns)

In [ ]:
for col in df.columns:
    print(col)

# Step 2 (cont.) : Select the 9 Task 2 entity categories

The load / column-clean / `_Answer`-rename cells above are reused from Task 1 unchanged. Task 1 *excluded* these fields; Task 2 trains on exactly them. `Filename` is metadata (it never appears in clause text), so it is **not** a Task 2 category. One assert per category catches schema drift immediately.

In [ ]:
task2_categories = [
    "Document Name", "Parties", "Agreement Date", "Effective Date",
    "Expiration Date", "Renewal Term", "Notice Period To Terminate Renewal",
    "Governing Law", "Warranty Duration",
]
assert len(task2_categories) == 9
for c in task2_categories:  # every category must have both a context and an answer column
    assert c in df.columns and f"{c}_Answer" in df.columns, f"missing column pair for {c}"
print(f"{len(task2_categories)} Task 2 categories:")
for c in task2_categories:
    print(" -", c)

In [ ]:
def save_jsonl(data, filename):
    with open(filename, 'w') as f:
        for entry in data:
            f.write(json.dumps(entry) + '\n')

# Step 3 : Normalize raw answer cells into clean values

The Task 2 counterpart of Task 1's `to_binary()` — one function that turns whatever is in an `_Answer` cell into a clean Python value the rest of the pipeline can trust. Three messy cases:

1. empty / NaN → `None` (entity not found),
2. bracketed list-strings like `"['5/8/2014']"` → unwrap (never fed through verbatim),
3. semicolon-separated multi-values (`"Party A; Party B"`) → list of strings.

In [ ]:
import ast

def normalize_answer(raw):
    """Raw _Answer cell -> None | str | list[str]."""
    if pd.isna(raw) or not str(raw).strip():
        return None
    s = str(raw).strip()
    # Case: bracketed list-string like "['5/8/2014']" or "['A', 'B']"
    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                parsed = [str(p).strip() for p in parsed if str(p).strip()]
                if not parsed:
                    return None
                s = "; ".join(parsed)      # fall through to semicolon handling
        except (ValueError, SyntaxError):
            pass                            # not a real list-string; keep as-is
    # Case: multi-value -> list of strings; single value -> plain string
    parts = [p.strip() for p in s.split(";") if p.strip()]
    if not parts:
        return None
    return parts if len(parts) > 1 else parts[0]

# Quick self-test on the documented cases
assert normalize_answer(None) is None
assert normalize_answer("  ") is None
assert normalize_answer("['5/8/2014']") == "5/8/2014"
assert normalize_answer("Party A; Party B") == ["Party A", "Party B"]
assert normalize_answer("Nevada") == "Nevada"
print("normalize_answer OK")

# Step 4 & 5 : Build extraction examples, split by contract

One example per (contract, category) where a clause excerpt exists: an instruction naming the entity, the clause text as input, and the JSON answer as output. Design decisions, stated explicitly:

- **Positive example** — context and answer both present: output is `{"<Category>": "<value>"}` (or a JSON list for multi-values).
- **"Not found" example** — context present but the normalized answer is `None`: output is `{"<Category>": null}`. This teaches the model to say "not there" *in-format* instead of hallucinating a value — the Task 2 analogue of Task 1 keeping its "No" examples.
- **Skip** when there is no context at all — with no clause text there is nothing to extract from.
- The instruction states the full JSON contract verbatim (single key, list for multi-values, `null` for absent) so the model is never left to guess the schema.

The split is done on **contracts (df rows) first**, 85/15, then examples are built from each side — same leakage rule as Task 1: no contract's clauses may appear in both sets.

In [ ]:
# Return the clause text in a column, or None if that cell is empty/blank.
def nonempty_context(row, cat):
    v = row.get(cat)
    return str(v).strip() if pd.notna(v) and str(v).strip() else None

# GOAL OF THIS FUNCTION:
# Turn a table of contracts into individual training examples for Task 2.
# Each example is one extraction question: "what is the <category> in THIS
# clause? Answer as single-key JSON". json.dumps handles str, list AND None
# (-> null) uniformly, so the output label is always valid JSON by construction.
def build_examples(frame):
    rows = []
    for _, row in frame.iterrows():          # one contract at a time
        for category in task2_categories:    # one question per entity category
            context = nonempty_context(row, category)
            if not context:
                continue                     # nothing to extract from -> skip
            value = normalize_answer(row[f"{category}_Answer"])
            rows.append({
                "instruction": (
                    f'Extract the "{category}" from the contract text below. '
                    f'Return the result as a JSON object with the single key '
                    f'"{category}". If multiple values exist, return them as a '
                    f'list of strings. If the value is not present, use null.'
                ),
                "category": category,
                "input": context,
                # json.dumps handles str, list AND None -> null uniformly
                "output": json.dumps({category: value}, ensure_ascii=False),
            })
    return rows

# Split by CONTRACT first, then build examples from each side,
# so no single contract's clauses end up in both train and validation.
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_data = build_examples(train_df)
val_data = build_examples(val_df)

print(f"Contracts — train: {len(train_df)}, val: {len(val_df)}")
print(f"Examples  — train: {len(train_data)}, val: {len(val_data)}")

In [ ]:
# Show one example of each shape: single value, list value, and null.
def first_where(data, pred):
    return next((e for e in data if pred(e)), None)

samples = {
    "SINGLE VALUE": first_where(train_data, lambda e: isinstance(json.loads(e["output"])[e["category"]], str)),
    "LIST VALUE":   first_where(train_data, lambda e: isinstance(json.loads(e["output"])[e["category"]], list)),
    "NULL VALUE":   first_where(train_data, lambda e: json.loads(e["output"])[e["category"]] is None),
}
for name, ex in samples.items():
    print(f"--- {name} EXAMPLE ---")
    if ex is None:
        print("(none found)")
    else:
        shown = dict(ex, input=ex["input"][:300] + ("..." if len(ex["input"]) > 300 else ""))
        print(json.dumps(shown, indent=2, ensure_ascii=False))
    print()

# Step 6 : Sanity checks — per-category counts + every output must round-trip `json.loads`

Two cheap checks that catch silent data bugs before an expensive training run (the Task 2 version of Task 1's Step 6a):

1. **Per-category counts** — every one of the 9 categories should contribute examples; a zero means a renamed/missing column.
2. **Every output must round-trip through `json.loads`** with the category as its only key — if a single training label is malformed JSON, the model is being *taught* to emit bad JSON.

> **No class balancing step.** Task 1 needed hard negatives + Yes/No balancing because it was a binary classifier. Task 2 has no majority class to collapse into — this per-category count check is its analogue.

In [ ]:
from collections import Counter

counts_train = Counter(e["category"] for e in train_data)
counts_val = Counter(e["category"] for e in val_data)
null_counts = Counter(e["category"] for e in train_data
                      if json.loads(e["output"])[e["category"]] is None)

print(f"{'Category':45s} {'train':>6s} {'val':>5s} {'null(train)':>12s}")
for c in task2_categories:
    print(f"{c:45s} {counts_train.get(c, 0):6d} {counts_val.get(c, 0):5d} {null_counts.get(c, 0):12d}")
    assert counts_train.get(c, 0) > 0, f"no train examples for {c}"
    assert counts_val.get(c, 0) > 0, f"no val examples for {c}"

# Every output label must be valid single-key JSON — otherwise we would be
# TEACHING the model to emit bad JSON.
for e in train_data + val_data:
    parsed = json.loads(e["output"])            # raises if malformed
    assert list(parsed.keys()) == [e["category"]], f"wrong key in {e['output']!r}"
print("\nAll outputs are valid single-key JSON.")

# Save examples to JSONL (ensure dirs exist; save paths == load paths)

Task-2-specific filenames so the artifacts never collide with the Task 1 JSONL.

In [ ]:
# All generated artifacts for this baseline go under OUTPUT_DIR (a dedicated
# subdirectory) so the fine-tuned Task 2 run's files are never overwritten. The
# validation JSONL written here is identical to the fine-tuned run's (same
# random_state=42 contract-level split / same pipeline).
CUAD_TRAIN_PATH = OUTPUT_DIR/'cuad'/'train'
CUAD_VALIDATION_PATH = OUTPUT_DIR/'cuad'/'validation'

# Ensure the train/ and validation/ directories exist before saving.
CUAD_TRAIN_PATH.mkdir(parents=True, exist_ok=True)
CUAD_VALIDATION_PATH.mkdir(parents=True, exist_ok=True)


In [ ]:
save_jsonl(train_data, CUAD_TRAIN_PATH/'cuad_task2_train.jsonl')
save_jsonl(val_data, CUAD_VALIDATION_PATH/'cuad_task2_validation.jsonl')
print(f"Saved {len(train_data)} training samples and {len(val_data)} validation samples.")

# Load the base model for inference (no training)

Instead of fine-tuning, load **`meta-llama/Meta-Llama-3.1-8B`** in 4-bit NF4 — exactly how the fine-tuning notebook loads its *base* model — then use it directly for generation. There is **no `SFTTrainer`, no `LoraConfig`, no completion-only loss, and no adapter saved to disk**. We only need the frozen base model to produce JSON extraction predictions on the validation set built above.


- Note : before execution of the cell below run to the terminal `$env:HF_TOKEN=your_hf_token`

In [ ]:
# Diagnostic: confirm WHICH account the token belongs to and whether it can access the gated repo.
# A 403 "not in the authorized list" means the token is valid but this account lacks access.
import os
from huggingface_hub import whoami, auth_check
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

def get_hf_token():
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv("HF_TOKEN")

hf_token = get_hf_token()
assert hf_token, "HF_TOKEN not found (Kaggle Secret or local .env)"

# Base model for the real (non-smoke-test) run. 8B in 4-bit NF4 is ~5-6 GB of weights,
# which fits fully in a single T4 (16 GB) VRAM with no CPU/disk offload. Smoke-test was "meta-llama/Llama-3.2-1B".
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B"

# 1) Which account is this token? Request access on the model page with THIS exact account.
me = whoami(token=hf_token)
print(f"Token belongs to: {me['name']}  (type: {me.get('type')})")

# 2) Does that account actually have access to the gated repo?
try:
    auth_check(MODEL_ID, token=hf_token)
    print(f"✅ Access granted to {MODEL_ID} — you can run the load cell below.")
except GatedRepoError:
    print(f"❌ Still gated for account '{me['name']}'.")
    print(f"   -> Visit https://huggingface.co/{MODEL_ID} while logged in as '{me['name']}', "
          f"accept the license, and wait for approval.")
    print(f"   -> Or use the ungated mirror: model_name = 'unsloth/Meta-Llama-3.1-8B'")
except HfHubHTTPError as e:
    print(f"❌ Auth/HTTP error (likely an invalid or expired token): {e}")


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
import os

def get_hf_token():
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    from dotenv import load_dotenv
    load_dotenv()  # reads .env from the current working dir (project root)
    return os.getenv("HF_TOKEN")

hf_token = get_hf_token()
assert hf_token, "HF_TOKEN not found (Kaggle Secret or local .env)"

from huggingface_hub import login
login(token=hf_token)

# 1. Configuration - base model, loaded AS-IS (NO fine-tuning, NO adapter).
# 8B in 4-bit NF4 is ~5-6 GB of weights, which fits fully in a single T4 (16 GB).
model_name = "meta-llama/Meta-Llama-3.1-8B"
MAX_SEQ_LENGTH = 1024   # matches the fine-tuned Task 2 run

# 2. QLoRA-style 4-bit load (same quantization the fine-tuned run used for its base model)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 3. Load Base Model (whole model on GPU 0; GPU 1 hidden in cell 1)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=hf_token
)

# 4. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loaded base model '{model_name}' in 4-bit for inference (no training).")


# Evaluate the base model on validation — JSON validity, exact match, token-level F1 (per category)

**The evaluation code below is the exact same code as the fine-tune notebook's eval cell** — same greedy generation (`do_sample=False`, `max_new_tokens=128`), same `parse_prediction` / `norm` / `token_f1` / `value_f1`, same per-category + overall aggregation (full logic: [TASK2_EVALUATION_METRICS.md](docs/task_2/TASK2_EVALUATION_METRICS.md), [TASK2_LOGIC.md](docs/task_2/TASK2_LOGIC.md) §3). Only the model differs: here it is the **un-fine-tuned base model**, so these are the **baseline** numbers.

Comparing them against the fine-tuned notebook's metrics on this identical validation set shows exactly what fine-tuning bought, metric by metric:

1. **JSON validity** — did the base model obey the strict single-key JSON contract at all? (Expected to be the largest delta — base models fence/chat/add keys.)
2. **Exact match** — of well-formed outputs, is the extracted value exactly the normalized gold value?
3. **Token-level F1** — partial credit; a high baseline F1 with low validity would mean the base model finds the right span but lacks format discipline.

Results are written to the **`no_finetune_baseline_task2/` subdirectory** (`eval_metrics.json`, `eval_report.txt`) so they sit alongside — but never overwrite — the fine-tuned run's artifacts.


In [ ]:
# Use cache for faster generation at inference time.
model.config.use_cache = True
model.eval()

def predict_json(example):
    """Greedy-generate the model's raw completion for one validation example."""
    prompt = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

def parse_prediction(raw_text, category):
    """Metric 1. Returns (value, is_valid). Valid iff json.loads succeeds AND the
    result is an object whose only key is exactly the requested category."""
    try:
        obj = json.loads(raw_text)
    except json.JSONDecodeError:
        return None, False
    if not isinstance(obj, dict) or set(obj.keys()) != {category}:
        return None, False
    return obj[category], True

def norm(v):
    """Light normalization for exact match: lowercase + strip; lists become
    order-insensitive (sorted tuples) — the order of parties is not meaningful."""
    if v is None:
        return None
    if isinstance(v, list):
        return tuple(sorted(str(x).strip().lower() for x in v))
    return str(v).strip().lower()

def token_f1(pred, gold):
    """SQuAD-style word-overlap F1 between two scalar values."""
    p, g = str(pred).lower().split(), str(gold).lower().split()
    common = Counter(p) & Counter(g)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / len(p), overlap / len(g)
    return 2 * precision * recall / (precision + recall)

def value_f1(pred, gold):
    """Token F1 extended to None and list values. For lists: greedily align each
    gold item to its best-matching predicted item; unmatched items on either
    side count as 0 (divide by max(len(pred), len(gold)))."""
    if gold is None and pred is None:
        return 1.0            # correctly said "not there"
    if gold is None or pred is None:
        return 0.0            # hallucinated a value, or missed a present one
    pred_list = pred if isinstance(pred, list) else [pred]
    gold_list = gold if isinstance(gold, list) else [gold]
    used, scores = set(), []
    for g in gold_list:
        best, best_j = 0.0, None
        for j, p in enumerate(pred_list):
            if j in used:
                continue
            s = token_f1(p, g)
            if s > best:
                best, best_j = s, j
        if best_j is not None:
            used.add(best_j)
        scores.append(best)
    return sum(scores) / max(len(pred_list), len(gold_list))

results = []
for i, ex in enumerate(val_data):
    raw = predict_json(ex)
    gold = json.loads(ex["output"])[ex["category"]]
    pred, valid = parse_prediction(raw, ex["category"])
    results.append({
        "category": ex["category"],
        "json_valid": valid,
        "exact_match": bool(valid and norm(pred) == norm(gold)),
        "f1": value_f1(pred, gold) if valid else 0.0,
    })
    if (i + 1) % 100 == 0:
        print(f"  evaluated {i + 1}/{len(val_data)}")

# Aggregate per category (all 9 rows, in the canonical order), then overall.
rdf = pd.DataFrame(results)
per_cat = (rdf.groupby("category")[["json_valid", "exact_match", "f1"]]
              .mean().reindex(task2_categories))
overall = rdf[["json_valid", "exact_match", "f1"]].mean()

header = f"{'Category':45s} {'json_valid':>10s} {'exact_match':>12s} {'token_f1':>9s}"
lines = [header]
for cat, row in per_cat.iterrows():
    lines.append(f"{cat:45s} {row['json_valid']:10.2f} {row['exact_match']:12.2f} {row['f1']:9.2f}")
lines.append("-" * len(header))
lines.append(f"{'OVERALL':45s} {overall['json_valid']:10.2f} {overall['exact_match']:12.2f} {overall['f1']:9.2f}")
report_str = "\n".join(lines)
print("Task 2 metrics on validation (BASELINE, no fine-tuning):\n")
print(report_str)

# --- Persist evaluation results as downloadable artifacts ---
# Written under OUTPUT_DIR (no_finetune_baseline_task2/) so `kaggle kernels output`
# pulls them back into a subfolder that never overwrites the fine-tuned run's metrics.
eval_metrics = {
    "model_name": model_name,
    "fine_tuned": False,
    "task": "task2_entity_extraction",
    "n_validation_examples": len(results),
    "overall": {k: float(v) for k, v in overall.items()},
    "per_category": {
        cat: {k: float(v) for k, v in row.items()}
        for cat, row in per_cat.iterrows()
    },
    "per_category_counts": rdf["category"].value_counts().to_dict(),
}
eval_metrics_path = OUTPUT_DIR / "eval_metrics.json"
with open(eval_metrics_path, "w", encoding="utf-8") as f:
    json.dump(eval_metrics, f, indent=2)

eval_report_path = OUTPUT_DIR / "eval_report.txt"
with open(eval_report_path, "w", encoding="utf-8") as f:
    f.write("Task 2 — JSON validity / exact match / token F1 on validation (BASELINE, no fine-tuning):\n\n")
    f.write(report_str + "\n")

print(f"\nWrote eval metrics to {eval_metrics_path}")
print(f"Wrote eval report  to {eval_report_path}")